### Feature Importance

> "Backtesting is not a research tool. Feature importance is."
>
> &mdash; Marco's first law of backtesting

This notebook will cover exercise answer.

* Exercise 8.1
* Exercise 8.2
* Exercise 8.3

As we go along, there will be some explanations.

Most of the functions below can be found under Tool/metrics

This chapter itself can be quite heavy as it involves PCA as well as different metric performance measures.

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cqrlib as rs

%matplotlib inline

In [ ]:
from sklearn.datasets import make_classification

def test_data(n_features=40, n_informative=10, n_redundant=10, n_samples=10000):
    # generate a random dataset for a classification problem    
    trnsX, cont = make_classification(n_samples=n_samples, n_features=n_features, n_informative=n_informative, n_redundant=n_redundant, random_state=0, shuffle=False)
    df0 = pd.date_range(periods=n_samples, freq=pd.tseries.offsets.BDay(), end=pd.datetime.today())
    trnsX = pd.DataFrame(trnsX, index=df0)
    cont = pd.Series(cont, index=df0).to_frame('bin')
    df0 = ['I_%s' % i for i in range(n_informative)] + ['R_%s' % i for i in range(n_redundant)]
    df0 += ['N_%s' % i for i in range(n_features - len(df0))]
    trnsX.columns = df0
    cont['w'] = 1.0 / cont.shape[0]
    cont['t1'] = pd.Series(cont.index, index=cont.index)
    return trnsX, cont

X, y = test_data(n_features=20, n_informative=5, n_redundant=5, n_samples= 3000)

#Take note of depreciation warning Pandas 1.0.3

In [ ]:
X

In [ ]:
y #look familiar?

In [ ]:
X0 = pd.DataFrame(data = rs.o_feat(X), index = X.index).add_prefix("PC_") 

#always make sure your columns are str only when running mp

**Note**

If PCA is new or you are confused. I think this article can further illustrate what we are doing.

[Towards Data Science: PCA using sklearn](https://towardsdatascience.com/pca-using-python-scikit-learn-e653f8989e60)

While you are running SFI method using multiprocessing for parallelization, it will improve processing speed greatly but not as much as you think. Its highly dependent on your machine.

For sklearn version 0.23.1, there is some base class and utilities which we can use to improve processing speed.

The below is one of them (Haven't tried it, but it seems legit):

[sklearn/feature_selection/variance_threshold](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html#sklearn.feature_selection.VarianceThreshold)

In [ ]:
# look at the time require to run 14 columns

rs.feat_imp_analysis(X = X0, 
                     y = y, 
                     sample_weight=y['w'],
                     methods = ['MDI', 'MDA', 'SFI'],
                     events = None,
                     pct_embargo = 0.01,
                     n_splits = 10,
                     min_weight_fraction_leaf = 0.0,
                     n_jobs = 3,
                     scoring = "accuracy",
                     output_path = None,
                     mask_effect = False) # masking effect is False: max_feature = int(1)

### PCA as an unsupervised ML

**MDI:** PC_3, PC_2, PC_0, PC_1 are considered important.

**MDA:** PC_2, PC_3, PC_0, PC_1 are considered important.
    
**SFI:** PC: PC_3, PC_2, PC_0 are considered important. (PC_1 was ranked below PC_12, PC_11)

With reference to the final outcome for the 3 methods: 3, 2, 0 are consistently at the top of the rank.

Firstly, we performed PCA on the initial 20 features to deduce 14 principle components (PC). This reduce dimensionality by dropping features with small eigen-values.

The resulting features/ PC, are considered more "principled" at explaining the structure of data/ samples.

**Conclusion**

This result definitely proves to be more reliable since unsupervised ML (PCA) has no idea on the labels (Less bias), reduced dimensionality of feature matrix proves to be less expensive.

In [ ]:
X0 = pd.concat([X0, X], axis=1) # same dataset so can reuse X

X0

In [ ]:
#now i have 34 columns instead of 14 columns, look at the time required when running with 4 cores only.

rs.feat_imp_analysis(X = X0, 
                     y = y, 
                     sample_weight=y['w'],
                     methods = ['MDI', 'MDA', 'SFI'],
                     events = None,
                     pct_embargo = 0.01,
                     n_splits = 10,
                     min_weight_fraction_leaf = 0.0,
                     n_jobs = 3,
                     scoring = "accuracy",
                     output_path = None,
                     mask_effect = False) 

### Mixing features with principle components

**MDI Summary:**

PC: PC_3,PC_2,PC_0, PC_1 are still considered important.

Features: I_1, R_0, I_4, I_0, I_2, R_1, R_2, R_3 are considered important.

For MDI, subsitution effect can be notice as principle components suffered a drop in ranking, however MDI still considered these PCs important. All PCs that was deemed important from initial test are still considered important.

While new features that were informative or redundant were mostly ranked important and above noise features.

With exception to R_4, I_3 which missed marginally from the dotted vertical red line.

**MDA Summary:**

PC: PC_3, PC_0, PC_4, PC_5 only were found above first noise features (N_4).

Features:  I_4, I_1, I_2, R_0, I_3 were still found above first noise features (N_4).

In view of substitution effects, MDA deem critical features as redundant and start to rank noise features higher. When dealing with critical but identical features, MDA will considered them unimportant (Since MDA based performance on Out-of-Sample score). 

As a result PC_2, PC_1 as well as the remaining informative and redundant features were mostly ranked below first noise feature (N_4). 

To make it worse, some of these critical features were deemed unimportant and deterimental to model.

**SFI Summary:** 

PC: PC_3, PC_2, PC_0 are ranked before then first noise feature (N_4) while PC_1 was ranked below N_8 (lower than N_4).

Features: I_3 was the only one that was ranked below first noise feature (N_4), while the remaining informative and redundant features was still above the first noise feature (N_4). (SFI also based performance on Out-of-Sample score).

However, SFI was the only method that ranked first principle feature (I_1) above principle components at the top.

### Conclusion

When features collection suffers high redundancy, substitution effects will reflect in the inconsistent ranking outcome from MDA against the other 2 methods.

With reference to the above, only the below labels was considered important in all 3 methods.

* PC_0
* PC_3
* I_4
* I_1
* I_2

Whenever strong mismatch outcome is observed. 

It is a sign where features collection is suffering from high feature redundancy.

In [ ]:
X1 = X0.drop(['PC_0', 'PC_3', 'I_4', 'I_1', 'I_2'], axis = 'columns')

In [ ]:
X1

In [ ]:
# dropped important labels while using remaining labels to run feat importance.

rs.feat_imp_analysis(X = X1, 
                     y = y, 
                     sample_weight=y['w'],
                     methods = ['MDI', 'MDA', 'SFI'],
                     events = None,
                     pct_embargo = 0.01,
                     n_splits = 10,
                     min_weight_fraction_leaf = 0.0,
                     n_jobs = 3,
                     scoring = "accuracy",
                     output_path = None,
                     mask_effect = False) 

### Removal of key features

After we removed the below important features/ PC: 

1. PC_0, PC_3
2. I_4, I_1, I_2

**MDI Summary:**

PC: PC_2, PC_1 are still considered important (Improvement in ranking).

All informative and redundant features were ranked above noise features, plus all these features (Including PC_2 and PC_1) were considered important.

**MDA Summary:**

PC: PC_2, PC_1 were found above first noise features (N_4).

Features:  I_3, R_2, R_0, I_0, R_4 were still found above first noise features (N_4).

All informative features (including PC_2 and PC_1) were ranked above first noise features. Significant improvement in overall ranking outcome. (Reduction in substitution effects)

**SFI Summary:** 

PC: only PC_2 was ranked before first noise feature (N_4) while PC_1 was still ranked below N_8 (lower than N_4).

Features: R_0, R_2, R_3, R_3, R_4 was the only one that was ranked below first noise feature (N_4), while the remaining informative and redundant features was still above the first noise feature (N_4).

I_3 still remained only one that was ranked below first noise feature (N_4), since SFI derive performance via OOS (Similar to MDA). Hence it can conclude all features are unimportant.

Overall, SFI has the least change/ improvement in terms of feature importance ranking, since SFI does not suffer from substitution effects since individual features are considered one at a time.

### Conclusion

After we removed the common important features among all 3 methods, there is a significant ranking improvement in informative features and principle components that were considered important.

As we removed the important labels, there was also a reduction in substitution effect across the samples. Hence reflecting an improvement in ranking outcome for informative features and key principle components.

With reference to SFI, the improvement seems limited as both I_3 and PC_1 were still considered "unimportant", after removing common "important" features. Both MDA and SFI based performance on OOS, therefore first principle features were able to be ranked at the top and important principle component can be considered "unimportant" (Below Noise features).

At the same time, overall OOB score and OOS score also reduced (This may mean in reduction in overall bias), variance on each features does reflect an increase (Bias vs Variance trade-off).

In short, whenever we believed that features collection are suffering from high substitution effects. Remove the common "important" labels may improve overall reliability of feature importance ranking (Bias reduction), however SFI on the other hand may not experience much improvement since this method is immuned substitution effects.

### Parallel vs Stacked features analysis

As seen above on the amount of time to run all the codes (even with parallelization).

> Reliable and efficient research result is a combination of 4 key factors:
> 1. your cognitive abilities
> 2. your curiosity
> 3. your data samples
> 4. your tools
>
> When it comes to efficiency, your tools probably takes the cake.
>
> &mdash; Undefeated's first principle of quantitative research

I will be skipping some exercises since my machine cannot handle such workload, you may refer to mlfinlab answers. 

It seems correct with my understanding.

Running parallel features analysis will give a big boost to speed, at the same time an increase in OOB and OOS score (Due to substitution effect, especially for large data set even with sample weight balanced) as compared to stacked feature analysis.

Since such ranking are derived from either OOS or OOB performance evaluation, if the score is unreliable (bias). It would be best to use stacked features.

**Note**

If you intend to run parallel feature analysis, only use mulitprocessing for parent process. Otherwise it will run into error, since daemon process cannot fork from child.